In [ ]:
# Ensure the proteus package is importable from the notebooks/ subdirectory
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

# Lab 4: Agent Execution & Evaluation

**Running Proteus Through Research Scenarios**

This is where everything comes together. You'll initialize the full agent harness,
run Proteus through multi-turn research scenarios, compare the engineered agent against
a naive baseline, and launch the Streamlit chat UI.

## Task 1: Restore state and initialize the agent

In [ ]:
import os, getpass

# Restore credentials
%store -r adb_dsn vector_user vector_password
os.environ["PROTEUS_DB_DSN"] = adb_dsn
os.environ["PROTEUS_DB_PASSWORD"] = vector_password
os.environ["PROTEUS_DB_USER"] = vector_user

openai_key = getpass.getpass("OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = openai_key

tavily_key = getpass.getpass("Tavily API Key (press Enter to skip): ")
if tavily_key:
    os.environ["TAVILY_API_KEY"] = tavily_key

In [ ]:
from proteus.db import connect_to_oracle
from proteus.vector_store import get_embedding_model, create_all_vector_stores
from proteus.memory_manager import MemoryManager
from proteus.toolbox import Toolbox
from proteus.tools import init_tools
from proteus.agent import init_agent, call_agent, call_agent_naive
from proteus.agent import context_size_history, naive_context_size_history, _naive_messages_by_thread
from proteus import config
from openai import OpenAI

vector_conn = connect_to_oracle()
embedding_model = get_embedding_model()
stores = create_all_vector_stores(vector_conn, embedding_model)

memory_manager = MemoryManager(
    conn=vector_conn,
    conversation_table=config.CONVERSATIONAL_TABLE,
    knowledge_base_vs=stores["knowledge_base_vs"],
    workflow_vs=stores["workflow_vs"],
    toolbox_vs=stores["toolbox_vs"],
    entity_vs=stores["entity_vs"],
    summary_vs=stores["summary_vs"],
    tool_log_table=config.TOOL_LOG_TABLE,
)

client = OpenAI()
toolbox = Toolbox(memory_manager=memory_manager, llm_client=client, embedding_model=embedding_model)
init_tools(memory_manager, toolbox, client, tavily_api_key=tavily_key or None)
init_agent(memory_manager, toolbox, client)

print("\n✅ Full agent stack initialized — Proteus is ready")

## Task 2: Run Proteus through research scenarios

### Scenario 1: Simple literature query

In [ ]:
call_agent(
    "I'm looking for papers on transformer architectures applied to time-series "
    "forecasting. What's in our knowledge base?",
    thread_id="SESSION-2025-001",
)

### Scenario 2: Follow-up on the same session

Watch how Proteus uses conversational memory from the previous turn:

In [ ]:
call_agent(
    "Interesting — can you search the web for more recent work on that topic? "
    "I want to see what's been published in 2025 or 2026.",
    thread_id="SESSION-2025-001",
)

### Scenario 3: New research thread requiring web search

In [ ]:
call_agent(
    "I've heard about a new technique called 'flow matching' for generative models. "
    "Can you find recent papers on it and summarize the key ideas?",
    thread_id="SESSION-2025-002",
)

### Scenario 4: Cross-referencing prior research

Proteus should recall entities and papers from previous sessions:

In [ ]:
call_agent(
    "Remember the papers we found on transformer architectures? "
    "How do those relate to the flow matching approach?",
    thread_id="SESSION-2025-003",
)

### Visualize context window growth

In [ ]:
import matplotlib.pyplot as plt

if context_size_history:
    tokens = [t for _, _, t in context_size_history]
    plt.figure(figsize=(8, 3))
    plt.plot(range(1, len(tokens) + 1), tokens, marker="o")
    plt.xlabel("Global Iteration (across all runs)")
    plt.ylabel("Estimated Tokens")
    plt.title("Context Window Size Over Agent Iterations")
    plt.tight_layout()
    plt.show()
else:
    print("No iterations recorded — run call_agent() first.")

---

## Task 3: Baseline — the naive agent (no context engineering)

`call_agent_naive` deliberately removes tool output offloading, summarization tools,
context refresh, and memory-backed context rebuild. Let's see what happens.

## Task 4: Head-to-head comparison

Five progressive queries that build on each other — tests memory continuity,
context management, and synthesis quality.

In [ ]:
import uuid

# Reset trackers
context_size_history.clear()
naive_context_size_history.clear()
_naive_messages_by_thread.clear()

eng_thread = str(uuid.uuid4())[:8]
naive_thread = str(uuid.uuid4())[:8]

queries = [
    "Search for recent papers on AI agent memory published in 2025 or 2026.",
    "Pick the most interesting paper from those results and give me the key takeaways.",
    "What other approaches or viewpoints might that paper have missed?",
    "Summarize everything we've discussed so far in this session.",
    "What was the first question I asked in this conversation?",
]

print("=" * 60)
print("ENGINEERED AGENT")
print("=" * 60)
for i, q in enumerate(queries, 1):
    print(f"\nQUERY {i}/5 >> {q}")
    call_agent(q, thread_id=eng_thread)

print("\n" + "=" * 60)
print("NAIVE AGENT (no context engineering)")
print("=" * 60)
for i, q in enumerate(queries, 1):
    print(f"\nQUERY {i}/5 >> {q}")
    call_agent_naive(q, thread_id=naive_thread)

### Visualize the difference

In [ ]:
import matplotlib.pyplot as plt

eng_tokens = [t for _, _, t in context_size_history]
naive_tokens = naive_context_size_history

plt.figure(figsize=(9, 4))
if eng_tokens:
    plt.plot(range(1, len(eng_tokens) + 1), eng_tokens, marker="o",
             label="With Context/Memory Engineering")
if naive_tokens:
    plt.plot(range(1, len(naive_tokens) + 1), naive_tokens, marker="s",
             label="Naive (no offloading/summarisation)")
plt.xlabel("Iteration")
plt.ylabel("Estimated Tokens")
plt.title("Context Window Growth: Engineered vs Naive Agent")
plt.legend()
plt.tight_layout()
plt.show()

---

## Task 5: Launch the Streamlit chat UI

Run the cell below to start Proteus as an interactive web app.
Open your browser to **http://localhost:8501** to chat with Proteus.

> **Note:** The Streamlit app will block this notebook cell while running.
> Press `Ctrl+C` or interrupt the kernel to stop it.

In [ ]:
# Launch the Streamlit chat UI
!streamlit run proteus/app.py --server.port 8501 --server.headless true

---

## Key Takeaways

**Agent Architecture Concepts:**
- An **agent run** is what `call_agent(...)` executes — one user turn handled
- Within a run, the **tool-call loop** repeats until a final answer
- The **agent harness** assembles context from memory, discovers tools, executes them, and persists results

**What makes the difference:**

| Aspect | Naive Agent | Proteus (Engineered) |
|--------|-------------|---------------------|
| **Context growth** | Unbounded | Managed — offloading + compaction |
| **Memory** | None — raw message history | 7 specialized types with semantic search |
| **Tool discovery** | All tools all the time | Semantic retrieval per query |
| **Knowledge retention** | Lost between sessions | Persistent across sessions |
| **Resolution patterns** | Rediscovered every time | Learned workflows reused |

---

## 🎉 Workshop Complete!

You've built a complete memory-powered AI research assistant:

| Lab | What You Built |
|-----|---------------|
| **1** | Deployed Oracle Autonomous DB and created VECTOR user |
| **2** | Vector search + memory architecture + MemoryManager + Toolbox |
| **3** | Context engineering + summarization + web search |
| **4** | Agent harness + scenarios + comparison + Streamlit UI |